# Limpeza do dataset Spotify

Notebook separado para remover registros considerados ruídos ou fora do escopo da análise.

Critérios aplicados:
- Faixas com menos de 1 min
- Faixas sem BPM (0 BPM)
- Faixas com speechiness maior que 0.5
- Faixas dos gêneros de ruído/ambientais e clássicos (comedy, children, opera, gospel, piano, romance, classical, show-tunes, kids, ambient...)
- Faixas com baixa popularidade (<= 10)
- Faixas com baixa loudness (<= -16)
- Faixas com nome contendo "rain sounds", "white noise", etc.

In [ ]:
# Instala pandas caso ainda não exista
try:
    import pandas as pd
except ModuleNotFoundError:
    !pip install pandas
    import pandas as pd

from pathlib import Path

base_dir = Path.cwd()
csv_path = base_dir / 'dataset' / 'dataset.csv'
output_path = base_dir / 'dataset' / 'dataset_cleaned.csv'

print(f'Arquivo de entrada: {csv_path}')
print(f'Arquivo de saída: {output_path}')

In [ ]:
df = pd.read_csv(csv_path)
print('Shape original:', df.shape)
print(df.head(3).to_string(index=False))
print('\nColunas:', list(df.columns))

In [ ]:
# Regras de remoção
undesirable_genres = {
    'comedy', 'children', 'opera', 'gospel', 'piano', 'romance',
    'classical', 'show-tunes', 'kids', 'ambient', 'new-age', 'soundtracks'
}

noise_keywords = [
    'rain sounds', 'white noise', 'nature sounds', 'sleep', 'meditation',
    'ambient', 'environmental sounds', 'soothing', 'soundscape'
]

# Normalização
df = df.copy()
df['track_genre'] = df['track_genre'].fillna('').astype(str).str.strip().str.lower()
df['track_name'] = df['track_name'].fillna('').astype(str).str.lower()

# Cria máscara de exclusão
mask = pd.Series(False, index=df.index)

mask |= df['duration_ms'] < 60000
mask |= df['tempo'] == 0
mask |= df['speechiness'] > 0.5
mask |= df['track_genre'].isin(undesirable_genres)
mask |= df['popularity'] <= 10
mask |= df['loudness'] <= -16

for keyword in noise_keywords:
    mask |= df['track_name'].str.contains(keyword, case=False, na=False)

# Totais por critério
criteria = {
    'duration_ms < 60000': df['duration_ms'] < 60000,
    'tempo == 0': df['tempo'] == 0,
    'speechiness > 0.5': df['speechiness'] > 0.5,
    'track_genre in blacklisted_genres': df['track_genre'].isin(undesirable_genres),
    'popularity <= 10': df['popularity'] <= 10,
    'loudness <= -16': df['loudness'] <= -16,
    'track_name contains noise keyword': False,
}

noise_mask = pd.Series(False, index=df.index)
for keyword in noise_keywords:
    noise_mask |= df['track_name'].str.contains(keyword, case=False, na=False)
criteria['track_name contains noise keyword'] = noise_mask

for name, cond in criteria.items():
    print(f'{name}: {int(cond.sum())} faixas removidas')

print(f'\nTotal de faixas removidas: {int(mask.sum())}')
print(f'Total restante: {int((~mask).sum())}')

In [ ]:
cleaned_df = df.loc[~mask].copy()
print('Shape final:', cleaned_df.shape)
print(cleaned_df.head(5).to_string(index=False))

In [ ]:
cleaned_df.to_csv(output_path, index=False)
print(f'Arquivo salvo em: {output_path}')

## Resultado da limpeza

Este notebook remove as faixas que se enquadram nos critérios definidos e salva um arquivo limpo em `dataset/dataset_cleaned.csv`.

Você pode usar esse dataset limpo em consultas SQL, análise exploratória ou treino de modelos.